# 07 — Comparativa Final: 3 Modelos

Genera todas las figuras comparativas para el Capítulo 4 del documento de grado.

**Flujo:**
1. Inferencia en test para U-Net++ (único modelo sin métricas test)
2. Cargar métricas de los 3 modelos
3. Generar figuras para el documento:
   - Matrices de confusión (3 modelos)
   - IoU por clase comparativo (3 modelos)
   - Resumen mIoU (val + test)
   - Curvas de convergencia
   - Predicciones comparativas (mismo patch, 3 modelos)
   - Tabla resumen CSV

**Artefactos generados en** `results/comparativa_tres_modelos/`

## 1. Setup

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
import json
from datetime import datetime
import pandas as pd

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.enabled = True

from scripts.utils import (
    get_device, set_seed,
    create_dataloaders, get_validation_augmentation,
    SegmentationMetrics, N_CLASSES, CLASS_NAMES,
    PATCHES_DIR, PATCHES_INDEX, CHECKPOINTS_DIR, RESULTS_DIR
)

set_seed(42)
device = get_device()

# Directorio de salida para todas las figuras comparativas
COMP_DIR = RESULTS_DIR / 'comparativa_tres_modelos'
COMP_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"Output dir: {COMP_DIR}")

## 2. Cargar DataLoader de Test (compartido para los 3 modelos)

In [ ]:
val_transform = get_validation_augmentation()

dataloaders = create_dataloaders(
    patches_dir=PATCHES_DIR,
    indices_csv=PATCHES_INDEX,
    batch_size=8,
    num_workers=0,
    train_transform=val_transform,
    val_transform=val_transform,
    use_balanced_sampler=False
)

test_loader = dataloaders['test']
print(f"Test set: {len(test_loader.dataset):,} patches — {len(test_loader)} batches")

## 3. Inferencia U-Net++ en Test Set

> U-Net y DeepLabv3+ ya tienen `test_metrics.json`. Este paso completa U-Net++.

In [ ]:
UNETPP_OUTPUT_DIR = RESULTS_DIR / 'test_evaluation' / 'unetplusplus_resnet50_baseline'
UNETPP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
UNETPP_METRICS_PATH = UNETPP_OUTPUT_DIR / 'test_metrics.json'

if UNETPP_METRICS_PATH.exists():
    print("U-Net++ test_metrics.json ya existe — saltando inferencia.")
    print("Borra el archivo y re-ejecuta si quieres recalcular.")
else:
    print("Ejecutando inferencia U-Net++ en test set...")

    # Cargar modelo
    unetpp_ckpt_path = CHECKPOINTS_DIR / 'unetplusplus_resnet50_baseline' / 'best_checkpoint.pth'
    assert unetpp_ckpt_path.exists(), f"Checkpoint no encontrado: {unetpp_ckpt_path}"

    model_unetpp = smp.UnetPlusPlus(
        encoder_name='resnet50',
        encoder_weights=None,
        in_channels=3,
        classes=N_CLASSES
    )
    ckpt = torch.load(unetpp_ckpt_path, map_location=device, weights_only=False)
    model_unetpp.load_state_dict(ckpt['model_state_dict'])
    model_unetpp = model_unetpp.to(device)
    model_unetpp.eval()

    print(f"Checkpoint: epoch {ckpt['epoch']}, val mIoU = {ckpt['best_val_miou']:.4f}")

    metrics_unetpp = SegmentationMetrics(num_classes=N_CLASSES, ignore_index=0, class_names=CLASS_NAMES)

    with torch.no_grad():
        for images, masks in tqdm(test_loader, desc="Inferencia U-Net++"):
            images = images.to(device)
            masks  = masks.to(device)
            outputs = model_unetpp(images)
            preds   = torch.argmax(outputs, dim=1)
            metrics_unetpp.update(preds, masks)

    summary_unetpp = metrics_unetpp.get_summary(detailed=True)
    metrics_unetpp.print_summary()

    # Guardar métricas
    iou_pp   = metrics_unetpp.get_iou_per_class()
    dice_pp  = metrics_unetpp.get_dice_per_class()
    prec_pp  = metrics_unetpp.get_precision_per_class()
    rec_pp   = metrics_unetpp.get_class_accuracy()
    f1_pp    = metrics_unetpp.get_f1_per_class()

    results_unetpp = {
        'model_name': 'unetplusplus_resnet50_baseline',
        'encoder': 'resnet50',
        'checkpoint_epoch': ckpt['epoch'],
        'val_miou': float(ckpt['best_val_miou']),
        'test_date': datetime.now().isoformat(),
        'test_patches': len(test_loader.dataset),
        'metrics': {
            'mIoU': float(summary_unetpp['mIoU']),
            'mIoU_with_bg': float(summary_unetpp['mIoU_with_bg']),
            'pixel_accuracy': float(summary_unetpp['pixel_accuracy'])
        },
        'per_class': {}
    }
    for i in range(N_CLASSES):
        results_unetpp['per_class'][CLASS_NAMES[i]] = {
            'IoU': float(iou_pp[i]),
            'Dice': float(dice_pp[i]),
            'Precision': float(prec_pp[i]),
            'Recall': float(rec_pp[i]),
            'F1': float(f1_pp[i])
        }

    with open(UNETPP_METRICS_PATH, 'w') as f:
        json.dump(results_unetpp, f, indent=2)

    np.save(UNETPP_OUTPUT_DIR / 'confusion_matrix.npy', metrics_unetpp.confusion_matrix)
    print(f"\nGuardado: {UNETPP_METRICS_PATH}")

    # Liberar GPU
    del model_unetpp
    torch.cuda.empty_cache()

## 4. Cargar métricas de los 3 modelos

In [ ]:
def load_metrics(model_folder):
    path = RESULTS_DIR / 'test_evaluation' / model_folder / 'test_metrics.json'
    with open(path) as f:
        return json.load(f)

m_unet    = load_metrics('unet_resnet34_baseline')
m_deeplab = load_metrics('deeplabv3_resnet50_baseline')
m_unetpp  = load_metrics('unetplusplus_resnet50_baseline')

# Etiquetas para figuras
MODEL_LABELS  = ['U-Net\n(ResNet-34)', 'DeepLabv3+\n(ResNet-50)', 'U-Net++\n(ResNet-50)']
MODEL_COLORS  = ['#3498db', '#e67e22', '#9b59b6']
CLASSES       = [CLASS_NAMES[i] for i in range(1, N_CLASSES)]   # sin Background

# Resumen rápido
print(f"{'Modelo':<30} {'Val mIoU':>10} {'Test mIoU':>10} {'Pixel Acc':>10}")
print("-" * 62)
for label, m in zip(['U-Net (ResNet-34)', 'DeepLabv3+ (ResNet-50)', 'U-Net++ (ResNet-50)'],
                     [m_unet, m_deeplab, m_unetpp]):
    print(f"{label:<30} {m['val_miou']:>10.4f} {m['metrics']['mIoU']:>10.4f} {m['metrics']['pixel_accuracy']:>10.4f}")

## 5. Figura 1 — mIoU comparativo (val + test)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

models_data = [
    ('U-Net\n(ResNet-34)',    m_unet['val_miou'],   m_unet['metrics']['mIoU'],   '#3498db'),
    ('DeepLabv3+\n(ResNet-50)', m_deeplab['val_miou'], m_deeplab['metrics']['mIoU'], '#e67e22'),
    ('U-Net++\n(ResNet-50)', m_unetpp['val_miou'],  m_unetpp['metrics']['mIoU'],  '#9b59b6'),
]

x = np.arange(len(models_data))
width = 0.35

vals  = [d[1] for d in models_data]
tests = [d[2] for d in models_data]
colors = [d[3] for d in models_data]

b1 = ax.bar(x - width/2, vals,  width, label='Validación', color=colors, alpha=0.6,
            edgecolor='black', linewidth=0.8, hatch='//')
b2 = ax.bar(x + width/2, tests, width, label='Test',       color=colors, alpha=0.95,
            edgecolor='black', linewidth=0.8)

ax.axhline(y=0.45, color='#27ae60', linestyle='--', alpha=0.8, linewidth=1.5, label='Meta ≥ 0.45')

for bar, val in zip(b1, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.004,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontstyle='italic')
for bar, val in zip(b2, tests):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.004,
            f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels([d[0] for d in models_data], fontsize=11)
ax.set_ylabel('mIoU', fontsize=12)
ax.set_title('Comparativa mIoU — 3 Modelos (Validación vs Test)', fontsize=13, fontweight='bold')
ax.set_ylim(0, 0.60)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(COMP_DIR / 'fig1_miou_comparativo.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"Guardado: {COMP_DIR / 'fig1_miou_comparativo.png'}")

## 6. Figura 2 — IoU por clase (3 modelos agrupados)

In [ ]:
iou_unet    = [m_unet['per_class'][c]['IoU']    for c in CLASSES]
iou_deeplab = [m_deeplab['per_class'][c]['IoU'] for c in CLASSES]
iou_unetpp  = [m_unetpp['per_class'][c]['IoU']  for c in CLASSES]

x = np.arange(len(CLASSES))
width = 0.26

fig, ax = plt.subplots(figsize=(15, 7))

b1 = ax.bar(x - width, iou_unet,    width, label=f'U-Net (mIoU={m_unet["metrics"]["mIoU"]:.4f})',
            color='#3498db', edgecolor='black', linewidth=0.6)
b2 = ax.bar(x,          iou_deeplab, width, label=f'DeepLabv3+ (mIoU={m_deeplab["metrics"]["mIoU"]:.4f})',
            color='#e67e22', edgecolor='black', linewidth=0.6)
b3 = ax.bar(x + width,  iou_unetpp,  width, label=f'U-Net++ (mIoU={m_unetpp["metrics"]["mIoU"]:.4f})',
            color='#9b59b6', edgecolor='black', linewidth=0.6)

ax.axhline(y=0.5, color='#27ae60', linestyle='--', alpha=0.7, linewidth=1.2, label='IoU = 0.5')
ax.axhline(y=0.3, color='#e74c3c', linestyle=':', alpha=0.6, linewidth=1.0, label='IoU = 0.3')

# Valores encima de barras (solo DeepLabv3+ para no saturar)
for bar, val in zip(b2, iou_deeplab):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
            f'{val:.3f}', ha='center', va='bottom', fontsize=7, fontweight='bold', color='#e67e22')

ax.set_xticks(x)
ax.set_xticklabels(CLASSES, rotation=35, ha='right', fontsize=11)
ax.set_ylabel('IoU', fontsize=12)
ax.set_title('IoU por Clase en Test Set — Comparativa 3 Modelos', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=10, loc='upper right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(COMP_DIR / 'fig2_iou_por_clase.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"Guardado: {COMP_DIR / 'fig2_iou_por_clase.png'}")

## 7. Figura 3 — Matrices de confusión (3 modelos)

In [ ]:
def load_cm(model_folder):
    path = RESULTS_DIR / 'test_evaluation' / model_folder / 'confusion_matrix.npy'
    cm = np.load(path).astype('float')
    # Normalizar por fila (recall)
    row_sums = cm.sum(axis=1, keepdims=True)
    cm_norm = np.where(row_sums > 0, cm / row_sums, 0.0)
    return cm_norm[1:, 1:]  # Sin Background

cm_unet    = load_cm('unet_resnet34_baseline')
cm_deeplab = load_cm('deeplabv3_resnet50_baseline')
cm_unetpp  = load_cm('unetplusplus_resnet50_baseline')

labels = [c.replace('CabeceraMunicipal', 'Cabecera') for c in CLASSES]

fig, axes = plt.subplots(1, 3, figsize=(22, 7))

for ax, cm, title in zip(
    axes,
    [cm_unet, cm_deeplab, cm_unetpp],
    [
        f'U-Net (ResNet-34)\nmIoU test = {m_unet["metrics"]["mIoU"]:.4f}',
        f'DeepLabv3+ (ResNet-50)\nmIoU test = {m_deeplab["metrics"]["mIoU"]:.4f}',
        f'U-Net++ (ResNet-50)\nmIoU test = {m_unetpp["metrics"]["mIoU"]:.4f}'
    ]
):
    sns.heatmap(
        cm, annot=True, fmt='.2f', cmap='Blues',
        xticklabels=labels, yticklabels=labels,
        ax=ax, vmin=0, vmax=1, linewidths=0.3,
        annot_kws={'size': 8}
    )
    ax.set_xlabel('Predicción', fontsize=10)
    ax.set_ylabel('Ground Truth', fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.tick_params(axis='y', rotation=0, labelsize=8)

plt.suptitle('Matrices de Confusión Normalizadas — Test Set', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(COMP_DIR / 'fig3_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"Guardado: {COMP_DIR / 'fig3_confusion_matrices.png'}")

## 8. Figura 4 — Curvas de convergencia (mIoU por época)

In [ ]:
def load_history(model_folder):
    path = CHECKPOINTS_DIR / model_folder / 'training_history.json'
    if path.exists():
        with open(path) as f:
            return json.load(f)
    return None

hist_unet   = load_history('unet_resnet34_baseline')
hist_deeplab = load_history('deeplabv3_resnet50_baseline')
hist_unetpp  = load_history('unetplusplus_resnet50_baseline')  # puede ser None

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- mIoU ---
ax = axes[0]
if hist_unet:
    ax.plot(hist_unet['val_miou'], label='U-Net (val)', color='#3498db', linewidth=1.8)
    ax.plot(hist_unet['train_miou'], label='U-Net (train)', color='#3498db', linewidth=1.0, linestyle='--', alpha=0.5)
if hist_deeplab:
    ax.plot(hist_deeplab['val_miou'], label='DeepLabv3+ (val)', color='#e67e22', linewidth=1.8)
    ax.plot(hist_deeplab['train_miou'], label='DeepLabv3+ (train)', color='#e67e22', linewidth=1.0, linestyle='--', alpha=0.5)
if hist_unetpp:
    ax.plot(hist_unetpp['val_miou'], label='U-Net++ (val)', color='#9b59b6', linewidth=1.8)
    ax.plot(hist_unetpp['train_miou'], label='U-Net++ (train)', color='#9b59b6', linewidth=1.0, linestyle='--', alpha=0.5)
else:
    # Marcar solo el mejor punto conocido de U-Net++
    best_unetpp_epoch = m_unetpp['checkpoint_epoch']
    best_unetpp_val   = m_unetpp['val_miou']
    ax.scatter([best_unetpp_epoch - 1], [best_unetpp_val], color='#9b59b6', s=120, zorder=5,
               label=f'U-Net++ (mejor val={best_unetpp_val:.4f}, época {best_unetpp_epoch})')

ax.axhline(y=0.45, color='#27ae60', linestyle=':', alpha=0.8, linewidth=1.2, label='Meta ≥ 0.45')
ax.set_xlabel('Época', fontsize=11)
ax.set_ylabel('mIoU', fontsize=11)
ax.set_title('Curvas de Convergencia — mIoU', fontsize=12, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# --- Loss ---
ax2 = axes[1]
if hist_unet:
    ax2.plot(hist_unet['val_loss'], label='U-Net (val)', color='#3498db', linewidth=1.8)
    ax2.plot(hist_unet['train_loss'], label='U-Net (train)', color='#3498db', linewidth=1.0, linestyle='--', alpha=0.5)
if hist_deeplab:
    ax2.plot(hist_deeplab['val_loss'], label='DeepLabv3+ (val)', color='#e67e22', linewidth=1.8)
    ax2.plot(hist_deeplab['train_loss'], label='DeepLabv3+ (train)', color='#e67e22', linewidth=1.0, linestyle='--', alpha=0.5)
if hist_unetpp:
    ax2.plot(hist_unetpp['val_loss'], label='U-Net++ (val)', color='#9b59b6', linewidth=1.8)
    ax2.plot(hist_unetpp['train_loss'], label='U-Net++ (train)', color='#9b59b6', linewidth=1.0, linestyle='--', alpha=0.5)

ax2.set_xlabel('Época', fontsize=11)
ax2.set_ylabel('Loss (Cross-Entropy)', fontsize=11)
ax2.set_title('Curvas de Convergencia — Loss', fontsize=12, fontweight='bold')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(COMP_DIR / 'fig4_curvas_convergencia.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"Guardado: {COMP_DIR / 'fig4_curvas_convergencia.png'}")

## 9. Figura 5 — Predicciones comparativas (mismo patch, 3 modelos)

Carga los 3 modelos en paralelo para visualizar el mismo patch con cada arquitectura.

In [ ]:
# Cargar los 3 modelos
def load_model(arch, encoder, ckpt_path):
    if arch == 'unet':
        m = smp.Unet(encoder_name=encoder, encoder_weights=None, in_channels=3, classes=N_CLASSES)
    elif arch == 'deeplabv3plus':
        m = smp.DeepLabV3Plus(encoder_name=encoder, encoder_weights=None, in_channels=3, classes=N_CLASSES)
    elif arch == 'unetplusplus':
        m = smp.UnetPlusPlus(encoder_name=encoder, encoder_weights=None, in_channels=3, classes=N_CLASSES)
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    m.load_state_dict(ckpt['model_state_dict'])
    m = m.to(device)
    m.eval()
    return m

print("Cargando 3 modelos...")
model_u    = load_model('unet',          'resnet34', CHECKPOINTS_DIR / 'unet_resnet34_baseline/best_checkpoint.pth')
model_d    = load_model('deeplabv3plus', 'resnet50', CHECKPOINTS_DIR / 'deeplabv3_resnet50_baseline/best_checkpoint.pth')
model_upp  = load_model('unetplusplus',  'resnet50', CHECKPOINTS_DIR / 'unetplusplus_resnet50_baseline/best_checkpoint.pth')
print("3 modelos cargados.")

In [ ]:
# Tomar primeros N patches del test set para visualización
N_SHOW = 6  # filas

sample_imgs, sample_masks_gt = [], []
for images, masks in test_loader:
    for i in range(min(images.shape[0], N_SHOW - len(sample_imgs))):
        sample_imgs.append(images[i])
        sample_masks_gt.append(masks[i])
    if len(sample_imgs) >= N_SHOW:
        break

# Inferencia con los 3 modelos
batch = torch.stack(sample_imgs).to(device)
with torch.no_grad():
    pred_u   = torch.argmax(model_u(batch),   dim=1).cpu()
    pred_d   = torch.argmax(model_d(batch),   dim=1).cpu()
    pred_upp = torch.argmax(model_upp(batch), dim=1).cpu()

# Colormap por clase
CMAP = plt.get_cmap('tab10')

# Figura: N_SHOW filas × 5 columnas (imagen | GT | U-Net | DeepLabv3+ | U-Net++)
fig, axes = plt.subplots(N_SHOW, 5, figsize=(20, 4 * N_SHOW))
col_titles = ['Imagen', 'Ground Truth', 'U-Net\n(ResNet-34)', 'DeepLabv3+\n(ResNet-50)', 'U-Net++\n(ResNet-50)']

for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=11, fontweight='bold')

for row in range(N_SHOW):
    img_np = sample_imgs[row].permute(1, 2, 0).numpy()
    img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)

    axes[row, 0].imshow(img_np)
    axes[row, 1].imshow(sample_masks_gt[row].numpy(), cmap='tab10', vmin=0, vmax=8)
    axes[row, 2].imshow(pred_u[row].numpy(),   cmap='tab10', vmin=0, vmax=8)
    axes[row, 3].imshow(pred_d[row].numpy(),   cmap='tab10', vmin=0, vmax=8)
    axes[row, 4].imshow(pred_upp[row].numpy(), cmap='tab10', vmin=0, vmax=8)

    for col in range(5):
        axes[row, col].axis('off')

plt.suptitle('Predicciones Comparativas — 3 Modelos (Test Set)', fontsize=13, fontweight='bold', y=1.002)
plt.tight_layout()
plt.savefig(COMP_DIR / 'fig5_predicciones_comparativas.png', dpi=200, bbox_inches='tight')
plt.show()
print(f"Guardado: {COMP_DIR / 'fig5_predicciones_comparativas.png'}")

# Liberar GPU
del model_u, model_d, model_upp
torch.cuda.empty_cache()

## 10. Figura 6 — Tabla de métricas por clase (heatmap)

In [ ]:
# Construir DataFrame con IoU por clase para los 3 modelos
df_iou = pd.DataFrame(
    {
        'U-Net':      [m_unet['per_class'][c]['IoU']    for c in CLASSES],
        'DeepLabv3+': [m_deeplab['per_class'][c]['IoU'] for c in CLASSES],
        'U-Net++':    [m_unetpp['per_class'][c]['IoU']  for c in CLASSES],
    },
    index=CLASSES
)

# Añadir fila de mIoU
df_iou.loc['mIoU'] = [
    m_unet['metrics']['mIoU'],
    m_deeplab['metrics']['mIoU'],
    m_unetpp['metrics']['mIoU']
]

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    df_iou, annot=True, fmt='.3f', cmap='RdYlGn',
    vmin=0, vmax=1, linewidths=0.5, linecolor='gray',
    ax=ax, annot_kws={'size': 11}
)
ax.set_title('IoU por Clase — Test Set (3 Modelos)', fontsize=13, fontweight='bold')
ax.set_xlabel('Modelo', fontsize=11)
ax.set_ylabel('Clase', fontsize=11)
ax.tick_params(axis='x', labelsize=10)
ax.tick_params(axis='y', labelsize=10, rotation=0)

# Separador visual antes de la fila mIoU
ax.axhline(y=len(CLASSES), color='black', linewidth=2)

plt.tight_layout()
plt.savefig(COMP_DIR / 'fig6_iou_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"Guardado: {COMP_DIR / 'fig6_iou_heatmap.png'}")

## 11. Tabla resumen CSV (para el documento)

In [ ]:
rows = []

for model_key, m, enc, params in [
    ('U-Net (ResNet-34)',    m_unet,    'ResNet-34', '24.4M'),
    ('DeepLabv3+ (ResNet-50)', m_deeplab, 'ResNet-50', '26.7M'),
    ('U-Net++ (ResNet-50)', m_unetpp,  'ResNet-50', '49.0M'),
]:
    row = {
        'Modelo': model_key,
        'Encoder': enc,
        'Parámetros': params,
        'Val mIoU': round(m['val_miou'], 4),
        'Test mIoU': round(m['metrics']['mIoU'], 4),
        'Pixel Acc': round(m['metrics']['pixel_accuracy'], 4),
    }
    for c in CLASSES:
        row[f'IoU_{c}'] = round(m['per_class'][c]['IoU'], 4)
    rows.append(row)

df_resumen = pd.DataFrame(rows)
csv_path = COMP_DIR / 'tabla_resumen_tres_modelos.csv'
df_resumen.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f"Guardado: {csv_path}")
print()
print(df_resumen[['Modelo', 'Val mIoU', 'Test mIoU', 'Pixel Acc']].to_string(index=False))

## 12. Resumen de artefactos generados

In [ ]:
archivos = [
    ('fig1_miou_comparativo.png',         'mIoU val + test, 3 modelos'),
    ('fig2_iou_por_clase.png',            'IoU por clase agrupado, 3 modelos'),
    ('fig3_confusion_matrices.png',       'Matrices de confusión normalizadas, 3 modelos'),
    ('fig4_curvas_convergencia.png',      'Curvas mIoU y loss por época'),
    ('fig5_predicciones_comparativas.png','Predicciones mismo patch, 3 modelos'),
    ('fig6_iou_heatmap.png',              'Heatmap IoU por clase y modelo'),
    ('tabla_resumen_tres_modelos.csv',    'Tabla maestra de métricas'),
]

print("=" * 70)
print("ARTEFACTOS GENERADOS")
print(f"Directorio: {COMP_DIR}")
print("=" * 70)
for fname, desc in archivos:
    path = COMP_DIR / fname
    status = '✓' if path.exists() else '✗'
    size_kb = path.stat().st_size // 1024 if path.exists() else 0
    print(f"  [{status}] {fname:<45} {desc}  ({size_kb} KB)")
print()
print("TABLA MAESTRA:")
print(df_resumen[['Modelo','Val mIoU','Test mIoU','Pixel Acc']].to_string(index=False))
print("=" * 70)

In [ ]:
pe